# SASRec 학습 및 블록 구조 비교 (Paper vs Author)

저자 전처리 데이터셋(`ml-1m`, `Video`, `Beauty`, `Steam`)에서 논문 수식 기반 블록(`paper`)과 저자 공개 코드 기반 블록(`author`)을 동일한 시드/초기화/평가 조건으로 학습하고 비교합니다.

In [ ]:
# 1. 실행 환경 확인
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("런타임 > 런타임 유형 변경에서 GPU를 선택하는 것을 권장")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. 구현 브랜치 가져오기
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/yewonkim03dev/paper-implementations.git"
REPOSITORY_DIR = Path("/content/paper-implementations")
PROJECT_NAME = "sasrec"
BRANCH = "implement/sasrec"

if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

In [ ]:
# 3. 의존성 설치
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPOSITORY_DIR / "requirements.txt")],
    check=True,
)

In [ ]:
# 4. 프로젝트 경로 설정 및 기본 단위 테스트 확인
import os

PROJECT_DIR = REPOSITORY_DIR / PROJECT_NAME
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Working directory: {Path.cwd()}")
subprocess.run([sys.executable, "test.py"], cwd=PROJECT_DIR, check=True)

In [ ]:
# 5. 출력 경로 설정 (Google Drive 영구 저장 여부 선택)
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/paper-implementations") / PROJECT_NAME / "outputs"
else:
    OUTPUT_ROOT = Path("/content/outputs")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output root: {OUTPUT_ROOT}")

In [ ]:
# 6. 저자 전처리 데이터셋 다운로드 (ml-1m / video / beauty / steam)
from urllib.request import urlretrieve
from src.data import load_sasrec_interactions, split_user_histories

# (파일명, 논문 max_len, 논문 dropout, 평가 max_users)
DATASET_PRESETS = {
    "ml-1m": ("ml-1m.txt", 200, 0.2, None),
    "video": ("Video.txt", 50, 0.5, None),
    "beauty": ("Beauty.txt", 50, 0.5, None),
    "steam": ("Steam.txt", 50, 0.5, 10_000),
}

DATASET_KEY = "ml-1m"
DATA_FILENAME, DEFAULT_MAX_LEN, DEFAULT_DROPOUT, DEFAULT_EVAL_MAX_USERS = DATASET_PRESETS[DATASET_KEY]

DATA_DIR = Path("/content/data/sasrec")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / DATA_FILENAME
RAW_BASE_URL = "https://raw.githubusercontent.com/kang205/SASRec/master/data"

if not DATA_PATH.exists():
    urlretrieve(f"{RAW_BASE_URL}/{DATA_FILENAME}", DATA_PATH)

histories = load_sasrec_interactions(DATA_PATH)
split = split_user_histories(histories)
print(
    f"dataset={DATASET_KEY} ({DATA_FILENAME}) | "
    f"users={len(histories):,} | items={split.num_items:,} | "
    f"interactions={sum(len(v) for v in histories.values()):,}"
)

## 학습 설정

`EVAL_NEGATIVES`를 `"100"`으로 두면 원논문 평가 방식(정답 1 + 무작위 미관측 negative 100개)으로 평가하고, `"all"`로 바꾸면 전체 미관측 아이템 대상 Full Ranking으로 평가합니다.

In [ ]:
# 7. 실험 하이퍼파라미터 설정
SEED = 42
VARIANTS = ("paper", "author")
PADDING_MODE = "fixed"  # "fixed" 또는 "dynamic"

MAX_LEN = DEFAULT_MAX_LEN
HIDDEN_DIM = 50
NUM_BLOCKS = 2
NUM_HEADS = 1
DROPOUT = DEFAULT_DROPOUT

EPOCHS = 200
BATCH_SIZE = 128
LR = 1e-3
L2_EMB = 0.0
NUM_WORKERS = 0

EVAL_K = 10
EVAL_NEGATIVES = "100"  # 양의 정수 문자열(예: "100", "500") 또는 "all"
EVAL_INTERVAL = 20
EVAL_MAX_USERS = DEFAULT_EVAL_MAX_USERS

In [ ]:
# 8. (선택) TensorBoard 대시보드 열기 — 실시간 모니터링이 필요할 때 주석 해제
# %load_ext tensorboard
# %tensorboard --logdir {str(OUTPUT_ROOT / DATASET_KEY)}

In [ ]:
# 9. paper / author 모델 순차 학습 실행 (각각 독립 프로세스로 동일 시드 보장)
RUN_DIRS = {}

for variant in VARIANTS:
    run_dir = OUTPUT_ROOT / DATASET_KEY / variant / f"seed{SEED}"
    RUN_DIRS[variant] = run_dir

    train_command = [
        sys.executable,
        "-u",
        "train.py",
        "--data-path", str(DATA_PATH),
        "--block-variant", variant,
        "--padding-mode", PADDING_MODE,
        "--max-len", str(MAX_LEN),
        "--hidden-dim", str(HIDDEN_DIM),
        "--num-blocks", str(NUM_BLOCKS),
        "--num-heads", str(NUM_HEADS),
        "--dropout", str(DROPOUT),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--l2-emb", str(L2_EMB),
        "--num-workers", str(NUM_WORKERS),
        "--eval-k", str(EVAL_K),
        "--eval-negatives", str(EVAL_NEGATIVES),
        "--eval-interval", str(EVAL_INTERVAL),
        "--seed", str(SEED),
        "--output-dir", str(run_dir),
        "--no-progress",
    ]
    if EVAL_MAX_USERS is not None:
        train_command.extend(["--eval-max-users", str(EVAL_MAX_USERS)])

    print(f"\n========== [{variant.upper()}] ==========")
    process = subprocess.Popen(
        train_command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, train_command)

## 결과 비교 및 시각화

각 실험의 `metrics.json`을 불러와 `paper`와 `author`의 검증·테스트 지표 표와 학습 곡선을 비교하고, 저장된 `best.pt`로 Full Ranking 교차 평가도 함께 확인합니다.

In [ ]:
# 10. metrics.json 요약 비교 표 및 학습 곡선 시각화
import json
import matplotlib.pyplot as plt

results = {
    variant: json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    for variant, run_dir in RUN_DIRS.items()
}

header = (
    f"{'Variant':<10} {'Best Epoch':>10} "
    f"{'Valid HR@' + str(EVAL_K):>14} {'Valid NDCG@' + str(EVAL_K):>16} "
    f"{'Test HR@' + str(EVAL_K):>14} {'Test NDCG@' + str(EVAL_K):>16}"
)
print(header)
print("-" * len(header))
for variant, summary in results.items():
    print(
        f"{variant:<10} {summary['best_epoch']:>10d} "
        f"{summary['best_valid']['hr']:>14.4f} {summary['best_valid']['ndcg']:>16.4f} "
        f"{summary['test']['hr']:>14.4f} {summary['test']['ndcg']:>16.4f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for variant, summary in results.items():
    epochs = [row["epoch"] for row in summary["history"]]
    losses = [row["train_loss"] for row in summary["history"]]
    axes[0].plot(epochs, losses, label=variant)

    eval_rows = [row for row in summary["history"] if "valid_ndcg" in row]
    axes[1].plot(
        [row["epoch"] for row in eval_rows],
        [row["valid_ndcg"] for row in eval_rows],
        marker="o",
        label=variant,
    )

axes[0].set_title(f"{DATASET_KEY}: Training BCE Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title(f"{DATASET_KEY}: Validation NDCG@{EVAL_K}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel(f"NDCG@{EVAL_K}")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.tight_layout()
plot_path = OUTPUT_ROOT / DATASET_KEY / f"comparison_seed{SEED}.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved plot: {plot_path}")

In [ ]:
# 11. 저장된 best.pt로 Full Ranking(전체 미관측 아이템) 교차 평가
from src.training import evaluate, load_model_checkpoint

eval_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Full Ranking Test Evaluation (device={eval_device})")
for variant, run_dir in RUN_DIRS.items():
    best_model = load_model_checkpoint(run_dir / "best.pt", device=eval_device)
    full_test = evaluate(
        best_model,
        split,
        eval_device,
        mode="test",
        k=EVAL_K,
        num_negatives=None,
        batch_size=BATCH_SIZE,
        padding_mode=PADDING_MODE,
        seed=SEED,
        max_users=EVAL_MAX_USERS,
    )
    print(
        f"{variant:<10} | Full Ranking Test HR@{EVAL_K}={full_test.hr:.4f} "
        f"NDCG@{EVAL_K}={full_test.ndcg:.4f}"
    )